# Real-World MCP Demo: Oracle OCI Usage MCP Server

This notebook demonstrates a real-world Model Context Protocol (MCP) integration.

Instead of creating our own MCP server, we connect to a published Oracle OCI Usage MCP Server.

Architecture:

User → LangChain Agent → MCP Client → Oracle OCI Usage MCP Server → OCI Usage API

The MCP client communicates with the MCP server using JSON-RPC over STDIO.
The MCP server then communicates with OCI services over HTTPS.

**Main lesson:** You usually do not build the MCP server yourself. You consume an MCP server published by a vendor, another team, or a third party.

## Learning Objectives

By the end of this notebook, you will understand how to:

1. Configure an MCP client for a third-party MCP server.
2. Dynamically discover MCP tools.
3. Connect discovered tools to a LangChain agent.
4. Ask a natural-language OCI cost question.
5. Let the agent select and invoke an MCP tool.
6. Inspect the agent response.
7. Convert structured usage data into a pandas DataFrame.
8. Create a pivot table and CSV report.
9. Analyze cost by service.
10. Understand direct API integration versus MCP.

## Important Note

The exact Oracle MCP server package name, executable name, authentication configuration,
and tool response schema can change over time.

This notebook follows the architecture demonstrated in the lesson. Before running it,
verify the current Oracle MCP server documentation and package name.

The example below uses the server command:

`oracle.oci-usage-mcp-server`

and launches it through `uvx`.

# 1. Install Dependencies

We install:

- `langchain` — agent framework
- `langchain-openai` — OpenAI model integration
- `langchain-mcp-adapters` — connects LangChain agents to MCP tools
- `mcp` — MCP Python SDK
- `pandas` — data analysis
- `python-dotenv` — environment variable management

In [ ]:
%pip install -U langchain langchain-openai langchain-mcp-adapters mcp pandas python-dotenv

# 2. Configure Environment Variables

Never hard-code API keys in source code.

Create a `.env` file in the same directory as this notebook.

Example:

```text
OPENAI_API_KEY=your_openai_api_key
```

OCI authentication may also be required depending on the Oracle MCP server configuration.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY is not configured. "
        "Add it to your .env file or environment."
    )

print("OpenAI API key found.")

# 3. Import Required Libraries

In [ ]:
import asyncio
import json
import os
import pandas as pd

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient

# 4. Configure the Oracle OCI Usage MCP Server

The MCP client needs to know how to start the MCP server.

In this example:

- `command="uvx"` launches the package in an isolated environment.
- `args` specifies the MCP server package.
- `transport="stdio"` means communication happens through standard input/output.

The client does not directly call OCI REST APIs.

Python Application
→ MCP Client
→ JSON-RPC over STDIO
→ Oracle OCI Usage MCP Server
→ HTTPS
→ OCI Usage API

In [ ]:
mcp_client = MultiServerMCPClient(
    {
        "oci_usage": {
            "command": "uvx",
            "args": [
                "oracle.oci-usage-mcp-server"
            ],
            "transport": "stdio",
        }
    }
)

print("MCP client configured.")

# 5. Discover MCP Tools

The client asks the MCP server what tools it provides.

This corresponds to the MCP `tools/list` operation.

We do not hard-code the tool implementation in our application.
The MCP server provides available tool definitions dynamically.

In [ ]:
tools = await mcp_client.get_tools()

print(f"Discovered {len(tools)} MCP tool(s):\n")

for tool in tools:
    print(f"Tool name: {tool.name}")
    print(f"Description: {tool.description}")
    print("-" * 80)

# 6. Inspect Tool Schemas

Depending on the adapter and MCP server version, tool schemas may be available
through the LangChain tool object.

In [ ]:
for tool in tools:
    print(f"\nTool: {tool.name}")
    print("Input schema:")

    try:
        print(tool.args_schema.model_json_schema())
    except Exception:
        print(getattr(tool, "args_schema", "Schema unavailable"))

# 7. Initialize the LLM

The LLM acts as the reasoning engine.

It does not directly execute OCI API calls.
It decides whether an available MCP tool should be called and what arguments
should be passed.

In [ ]:
model = ChatOpenAI(
    model="gpt-5.5",
    temperature=0
)

print("LLM initialized.")

# 8. Create the LangChain Agent

In [ ]:
agent = create_agent(
    model=model,
    tools=tools
)

print("LangChain agent created successfully.")

# 9. Ask a Natural-Language OCI Cost Question

Example:

> Show me the cost for the last 30 days as a daily breakdown by service.

The agent should determine which MCP tool is appropriate and invoke it.

In [ ]:
question = """
Show me the cost for the last 30 days as a daily breakdown by service.
Provide the result in a clear and concise table.
"""

result = await agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question,
            }
        ]
    }
)

print("Agent execution completed.")

# 10. Display the Final Agent Response

In [ ]:
messages = result["messages"]
final_message = messages[-1]

print(final_message.content)

# 11. Inspect Agent Execution Messages

The execution history can contain:

- User message
- Assistant tool call
- Tool result
- Final assistant response

This is useful for debugging and understanding the agent loop.

In [ ]:
for index, message in enumerate(messages, start=1):
    print(f"\n--- Message {index} ---")
    print("Type:", type(message).__name__)
    print("Content:", getattr(message, "content", ""))

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool calls:")
        print(message.tool_calls)

# 12. Example Structured Usage Data

The exact output format of the Oracle MCP server can vary.

The following cell demonstrates the type of structured data we might transform
into a pandas DataFrame.

Replace this example with the actual structured usage data returned by your MCP server.

In [ ]:
usage_data = [
    {
        "date": "2026-05-05",
        "service": "Block Storage",
        "cost": 1300.00,
    },
    {
        "date": "2026-05-05",
        "service": "Database",
        "cost": 1400.00,
    },
    {
        "date": "2026-05-05",
        "service": "VMware",
        "cost": 1951.00,
    },
]

df = pd.DataFrame(usage_data)
df

# 13. Create a Pivot Table

In [ ]:
pivot_df = df.pivot_table(
    index="date",
    columns="service",
    values="cost",
    aggfunc="sum",
    fill_value=0,
)

pivot_df["Total"] = pivot_df.sum(axis=1)

pivot_df

# 14. Calculate Total Cost

In [ ]:
total_cost = pivot_df["Total"].sum()

print(f"Total cost for the selected period: ${total_cost:,.2f}")

# 15. Calculate Cost by Service

In [ ]:
service_totals = (
    df.groupby("service")["cost"]
    .sum()
    .sort_values(ascending=False)
)

service_totals

# 16. Export the Results to CSV

In [ ]:
output_file = "oci_usage_last_30_days.csv"

pivot_df.to_csv(output_file)

print(f"CSV report created: {output_file}")

# 17. Read the CSV Report

In [ ]:
csv_df = pd.read_csv(
    "oci_usage_last_30_days.csv",
    index_col=0,
)

csv_df

# 18. Visualize Cost by Service

In [ ]:
import matplotlib.pyplot as plt

service_totals.plot(
    kind="bar",
    figsize=(10, 5),
    title="OCI Cost by Service",
)

plt.xlabel("Service")
plt.ylabel("Cost")
plt.tight_layout()
plt.show()

# 19. Compare MCP Data with OCI Console

The validation process is:

1. Run the MCP agent for the selected date range.
2. Export the returned data to CSV.
3. Open OCI Console.
4. Navigate to Billing and Cost Management.
5. Open Cost Analysis.
6. Select the same date range.
7. Compare costs by service.

The numbers should match when the same date range, tenancy, currency, granularity,
and filters are used.

This validation confirms that the MCP server is correctly retrieving the underlying
OCI usage data.

# 20. Real-World Architecture

The complete flow is:

User
  ↓
Natural Language Query
  ↓
LangChain Agent + LLM
  ↓
MCP Client
  ↓ JSON-RPC / STDIO
Oracle OCI Usage MCP Server
  ↓ HTTPS
OCI Usage API
  ↓
Usage Data
  ↓
MCP Client
  ↓
LangChain Agent
  ↓
Final AI Response
  ↓
Console / CSV

The key distinction is that our application does not directly implement the OCI
Usage API integration.

# 21. Direct API Integration vs MCP

## Without MCP

Each application may need to implement:

- OCI authentication
- OCI API requests
- API schemas
- Error handling
- Usage API integration
- Data transformation

## With MCP

The MCP server provides a standardized interface.

The application:

1. Connects to the MCP server.
2. Discovers tools.
3. Calls tools.
4. Receives results.

The MCP server handles vendor-specific API integration.

# 22. Why This Is a Real-World MCP Use Case

The Math MCP Server was useful for learning because we controlled both sides.

In production, the MCP server may be created and maintained by:

- A cloud provider
- A SaaS vendor
- An internal platform team
- A third-party developer
- An enterprise software provider

The AI application does not need to understand the server's internal implementation.
It only needs to understand the MCP interface.

That is where MCP becomes particularly valuable.

# 23. Example Questions for the Agent

Once the OCI Usage MCP tool is available, users can ask:

- What was our total OCI cost over the last 30 days?
- Which service cost us the most?
- Show me a daily cost breakdown.
- Which services are increasing in cost?
- What are our biggest cost drivers?
- Compare this month's cost with last month's cost.
- Identify unusual cost spikes.
- Summarize our cloud spending trends.

The MCP server provides access to the data.
The LLM provides natural-language reasoning and analysis on top of that data.

# 24. Key Takeaways

### 1. We consume the MCP server
We do not write or maintain the OCI Usage MCP server.

### 2. The server abstracts the OCI API
Our application does not directly implement the OCI Usage API integration.

### 3. Tools are dynamically discovered
The MCP client discovers tools exposed by the server.

### 4. LangChain remains the agent framework
MCP does not replace the LLM or the agent loop.
It provides a standardized tool integration layer.

### 5. The same MCP server can support multiple hosts
A published MCP server can potentially be used by different AI applications
and agent frameworks.

### 6. MCP is especially useful for reusable external integrations
The more complex and reusable the underlying service is, the more valuable
a standardized MCP interface becomes.

# 25. Final Summary

This notebook demonstrated the real-world MCP pattern.

Unlike the earlier Math MCP example, we did not build both sides of the system.
Instead, we built the client and consumed a published MCP server.

The key architecture is:

**User → LLM Agent → MCP Client → Oracle MCP Server → OCI API**

The LLM reasons.
The LangChain agent orchestrates.
The MCP client communicates with the MCP server.
The MCP server handles vendor-specific integration.
The OCI API provides the underlying cloud usage data.

**Build once, publish the MCP integration, and let multiple AI applications consume it through a standardized protocol.**